# AI Risk Manager: Home Credit Model
Place `application_train.csv` and `application_test.csv` in `data/` before running.

In [ ]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import shap

DATA = Path('data')
train = pd.read_csv(DATA / 'application_train.csv')
test = pd.read_csv(DATA / 'application_test.csv')

def engineer(frame):
    frame = frame.copy()
    frame['DEBT_TO_INCOME'] = frame['AMT_CREDIT'] / frame['AMT_INCOME_TOTAL'].replace(0, np.nan)
    frame['CREDIT_TO_ANNUITY'] = frame['AMT_CREDIT'] / frame['AMT_ANNUITY'].replace(0, np.nan)
    frame['AGE_YEARS'] = (-frame['DAYS_BIRTH']) / 365.25
    frame['EMPLOYMENT_YEARS'] = frame['DAYS_EMPLOYED'].clip(upper=0).abs() / 365.25
    return frame.replace([np.inf, -np.inf], np.nan)

X = engineer(train.drop(columns=['TARGET']))
y = train['TARGET']
numeric = X.select_dtypes(include=['number']).columns.tolist()
categorical = X.select_dtypes(exclude=['number']).columns.tolist()
preprocessor = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical),
])
X_encoded = preprocessor.fit_transform(X)
X_balanced, y_balanced = SMOTE(random_state=42).fit_resample(X_encoded, y)
X_train, X_valid, y_train, y_valid = train_test_split(X_balanced, y_balanced, test_size=0.2, stratify=y_balanced, random_state=42)
model = LGBMClassifier(n_estimators=350, learning_rate=0.04, num_leaves=31, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)
probabilities = model.predict_proba(X_valid)[:, 1]
print('ROC-AUC:', roc_auc_score(y_valid, probabilities))
print(classification_report(y_valid, (probabilities >= 0.5).astype(int)))
explainer = shap.TreeExplainer(model)
with open('preprocessor.pkl', 'wb') as f: pickle.dump({'transformer': preprocessor, 'features': list(X.columns)}, f)
with open('risk_model.pkl', 'wb') as f: pickle.dump(model, f)
with open('shap_explainer.pkl', 'wb') as f: pickle.dump(explainer, f)